# 02 - Transformação

Aplica as decisões registradas no `01_eda.ipynb`.

In [ ]:
import sys
sys.path.append("..")

import src.functions as fn #Funcoes criadas no projeto
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Carrega cópias de trabalho mantendo os arquivos originais intactos.
cadastral = pd.read_parquet("../dados/base_cadastral.parquet").copy()
submissao = pd.read_parquet("../dados/base_submissao.parquet").copy()
emprestimos = pd.read_parquet("../dados/historico_emprestimos.parquet").copy()
parcelas = pd.read_parquet("../dados/historico_parcelas.parquet").copy()

print(
    f"cadastral: {cadastral.shape} | submissao: {submissao.shape} | "
    f"emprestimos: {emprestimos.shape} | parcelas: {parcelas.shape}"
)

In [ ]:
# 1. Identificação de datas de nascimento sentinela e conversão de tipos
# Marca datas anteriores a 1920 como anomalias de preenchimento
datas_convertidas = pd.to_datetime(cadastral["data_nascimento"], errors="coerce")
cadastral["data_nascimento_invalida"] = (
    cadastral["data_nascimento"].notna() & (datas_convertidas.dt.year <= 1920)
)
cadastral.loc[cadastral["data_nascimento_invalida"], "data_nascimento"] = None

# Aplica conversão padronizada de tipos estruturais
cadastral, rel_tipos = fn.converter_tipos(
    cadastral,
    datas={"data_nascimento": ["%Y-%m-%d", "%d/%m/%Y"]},
    numericas_texto={"renda_anual": {"decimal": ",", "milhar": "."}} if cadastral["renda_anual"].dtype == "object" else None,
    inteiros=["qtd_membros_familia"],
)

# 2. Padronização de variáveis categóricas e conversão de placeholders em NaN
cols_categoricas = [
    c for c in cadastral.select_dtypes(include=["object", "string", "category"]).columns 
    if c not in ["data_nascimento"]
]
cadastral, rel_cat = fn.padronizar_categoricas(cadastral, colunas=cols_categoricas)

# 3. Consolidação de duplicatas: mais recentes e fallback com preenchimento de nulos
mascara_dup = cadastral.duplicated(subset=["id_cliente"], keep=False)
cad_unicos = cadastral.loc[~mascara_dup].set_index("id_cliente")

cad_1a = cadastral.loc[cadastral.duplicated(subset=["id_cliente"], keep="last")].set_index("id_cliente")
cad_2a = cadastral.loc[cadastral.duplicated(subset=["id_cliente"], keep="first")].set_index("id_cliente")

# A 2ª ocorrência prevalece; onde houver NaN, herda o valor da 1ª ocorrência
cad_dups_consolidadas = cad_2a.combine_first(cad_1a)

cadastral_tratada = (
    pd.concat([cad_unicos, cad_dups_consolidadas])
    .sort_index()
    .reset_index()
)

# 4. Otimização de tipos finais para economia de memória
cadastral_tratada["id_cliente"] = cadastral_tratada["id_cliente"].astype("int64")
cadastral_tratada["qtd_filhos"] = cadastral_tratada["qtd_filhos"].astype("Int64")
for col in cols_categoricas:
    cadastral_tratada[col] = cadastral_tratada[col].astype("category")

# 5. Asserções e validações do contrato de dados
assert cadastral_tratada.shape[0] == 50000, f"Esperado 50.000 linhas, obtido {cadastral_tratada.shape[0]}"
assert cadastral_tratada["id_cliente"].nunique() == 50000, "Falha: id_cliente não é único na base consolidada"

print("--- Base Cadastral Tratada com Sucesso ---")
print(f"Shape final: {cadastral_tratada.shape}")
print(f"Clientes únicos: {cadastral_tratada['id_cliente'].nunique()}")
print(f"Total de registros com data_nascimento_invalida = True: {cadastral_tratada['data_nascimento_invalida'].sum()}")
print("\nValores ausentes remanescentes por coluna:")
display(cadastral_tratada.isna().sum().loc[lambda s: s > 0])